# The Anatomy of AI · Lecture 1 · One Neuron
### The notebook

In the lecture we took apart the smallest working part of a neural network: one **neuron**, deciding whether to go for a
walk. Here you build that neuron yourself, from nothing but arithmetic, and check every number against the lecture.

The notebook follows the lecture's plates in order. Each section names its plate. Wherever a result appears in the
lecture, a `check(...)` line compares yours with it and prints ✓. If a check fails, the notebook stops.

There are five **Your turn** exercises along the way. Worked solutions are at the very end.

It runs on free Colab in under a minute, on the CPU. No GPU, no downloads, no keys.

In [ ]:
# Pinned to the versions free Colab ships (Python 3.12); nothing else is needed, and no keys.
%pip install -q numpy==2.0.2 matplotlib==3.10.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
})
INKMAP = LinearSegmentedColormap.from_list("ink", [PAPER, INK])   # a value as ink density, as on the plates


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "−")


def check(name, value, expected, tol=1e-9):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    want = f"{expected:.6g}" if isinstance(expected, float) else expected
    print(f"{'✓' if ok else '✗'} {name}: {shown}   (lecture: {want})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

## Plates I–III · One part, repeated

Every AI system is built from one small part, copied many times and wired together. That part is a **neuron**.
Big systems have billions of them. We take one apart completely.

The question of the whole course is **interpretability**: can we understand what a network is doing by looking inside it,
instead of just trusting what it says? A single neuron is where that question is easiest, so we start here.

(The idea is older than the computer: McCulloch and Pitts described a neuron like this one in 1943, and Rosenblatt built
a machine of them in 1958.)

## Plate IV · A decision

Our neuron decides: *should I go for a walk?* It looks at two numbers, its **inputs**, each between 0 and 1:

- **warmth**: 0.8, pleasantly warm;
- **rain**: 0.1, barely a drizzle.

It gives back one number, its **output**. On the plates, the darker a ball, the bigger its number.

In [ ]:
warmth, rain = 0.8, 0.1      # the sunny day

## Plate V · The weights

Each input comes in along a wire with a **weight**: a number that says how much that input counts, and which way.

- warmth has weight **+2.0**: warmth counts *for* a walk;
- rain has weight **−3.0**: rain counts *against* it, and more strongly.

On the plates a positive weight is a solid wire and a negative one is hollow; the thicker the wire, the more it counts.

In [ ]:
w_warmth, w_rain = 2.0, -3.0     # Lecture 1's walk neuron
bias = -0.5                      # explained on the next plate

## Plate VI · Step one: multiply and add

1. Multiply warmth by its weight.
2. Multiply rain by its weight. A negative weight pulls the sum down.
3. Add the **bias**: a fixed number the neuron always adds, here −0.5, a built-in reluctance.
4. Where the walk along the number line ends is the **total**.

In [ ]:
terms = {"warmth × 2.0": warmth * w_warmth, "rain × (−3.0)": rain * w_rain, "bias": bias}
total = sum(terms.values())
for name, t in terms.items():
    print(f"{name:>14}: {t:+.2f}")
print(f"{'total':>14}: {total:+.2f}")

check("warmth term", terms["warmth × 2.0"], 1.6)
check("rain term", terms["rain × (−3.0)"], -0.3)
check("total on the sunny day", total, 0.8)

Here is the same sum as the plate draws it: each term a step along the number line.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.4))
x = 0.0
for i, (name, t) in enumerate(terms.items()):
    y = 2 - i
    ax.annotate("", xy=(x + t, y), xytext=(x, y), arrowprops=dict(arrowstyle="-|>", color=INK, lw=2.2 if t > 0 else 1.2,
                linestyle="-" if t > 0 else "--"))
    ax.text(x + t / 2, y + 0.25, minus(f"{name}: {t:+.1f}"), ha="center")
    ax.plot([x + t, x + t], [y, y - 1], color=INK2, lw=0.6, ls=":")
    x += t
ax.plot([x], [-0.6], "o", color=INK)
ax.text(x, -1.05, minus(f"total {x:+.1f}"), ha="center")
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlim(-0.6, 2.0); ax.set_ylim(-1.4, 2.7); ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_xlabel("the number line")
plt.show()

## Plate VII · Step two: the gate

The total then goes through a **gate**:

1. a total below zero gives **0**: the neuron is **silent**;
2. a total above zero passes through unchanged: the neuron **fires**.

This gate is called **ReLU**: output = max(0, total).

In [ ]:
def relu(t):
    return np.maximum(0.0, t)

t = np.linspace(-2, 2, 401)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(t[t <= 0], relu(t[t <= 0]), color=INK, lw=1.2, ls="--", label="silent: output 0")
ax.plot(t[t >= 0], relu(t[t >= 0]), color=INK, lw=2.6, label="fires: output = total")
ax.set_xlabel("total"); ax.set_ylabel("output"); ax.legend(loc="upper left")
plt.show()

check("gate on −0.7", relu(-0.7), 0.0)
check("gate on 0.8", relu(0.8), 0.8)

## Plates VIII–IX · A sunny day, a rainy day

Put the two steps together and you have the whole neuron. We wrap it in a function, and a small drawing of it in the
lecture's style: inputs shaded by their value, wires as thick as their weight (hollow if negative), and the neuron shaded
by its output.

In [ ]:
def neuron(warmth, rain, w_warmth=2.0, w_rain=-3.0, bias=-0.5):
    """Lecture 1's neuron: multiply and add, then the gate. Returns (total, output)."""
    total = warmth * w_warmth + rain * w_rain + bias
    return total, float(relu(total))


def draw_neuron(warmth, rain, w_warmth=2.0, w_rain=-3.0, bias=-0.5, ax=None):
    total, out = neuron(warmth, rain, w_warmth, w_rain, bias)
    ax = ax or plt.subplots(figsize=(6.4, 3))[1]
    ax.set_xlim(0, 10); ax.set_ylim(0, 5); ax.set_aspect("equal"); ax.axis("off")
    for y, val, w, name in ((3.9, warmth, w_warmth, "warmth"), (1.1, rain, w_rain, "rain")):
        lw = 1.5 + 2.2 * abs(w)
        ax.plot([2.0, 4.95], [y, 2.5], color=INK, lw=lw, solid_capstyle="butt", zorder=1)
        if w < 0:                                    # hollow: a negative weight
            ax.plot([2.0, 4.95], [y, 2.5], color=PAPER, lw=lw - 3.2, solid_capstyle="butt", zorder=2)
        ax.add_patch(plt.Circle((1.5, y), 0.5, fc=INKMAP(val), ec=INK, lw=1.2, zorder=3))
        ax.text(0.85, y, f"{name}\n{val:g}", ha="right", va="center")
        ax.text(3.5, (y + 2.5) / 2 + (0.45 if y > 2.5 else -0.45), minus(f"× {w:+.1f}"), ha="center")
    ax.add_patch(plt.Circle((5.7, 2.5), 0.75, fc=INKMAP(min(1.0, out) * 0.8), ec=INK, lw=1.6, zorder=3))
    ax.text(5.7, 1.35, minus(f"bias {bias:+.1f}"), ha="center", va="top")
    ax.text(5.7, 3.45, minus(f"total {total:+.2f}"), ha="center")
    ax.plot([6.45, 8.0], [2.5, 2.5], color=INK, lw=2.4 if out > 0 else 1, ls="-" if out > 0 else ":")
    ax.text(8.15, 2.5, f"{out:.2g}", va="center", fontsize=14)
    return ax


fig, axes = plt.subplots(1, 2, figsize=(12, 3))
draw_neuron(0.8, 0.1, ax=axes[0]); axes[0].set_title("Plate VIII · a sunny day: fires")
draw_neuron(0.8, 0.6, ax=axes[1]); axes[1].set_title("Plate IX · a rainy day: silent")
plt.show()

total_sunny, out_sunny = neuron(0.8, 0.1)
total_rainy, out_rainy = neuron(0.8, 0.6)
check("sunny day: output", out_sunny, 0.8)
check("rainy day: total", total_rainy, -0.7)
check("rainy day: output", out_rainy, 0.0)

### Your turn 1 · How dry must it get?

On the rainy day the neuron is silent. Keep warmth at 0.8 and lower the rain. At what rain does the neuron start to fire
again?

*Hint:* the neuron starts to fire where its total is exactly 0. Write the total for warmth 0.8 and solve for rain.

In [ ]:
rain_edge = None   # TODO: the rain at which the total is exactly 0, for warmth 0.8

## Plate X · Every possible day at once

So far, two days. Now ask the neuron about *every* day: every warmth from 0 to 1 against every rain from 0 to 1. Each
column on the plate is one day, its height the neuron's output. The sunny day stands at 0.8; the rainy day is a pin at 0.

In [ ]:
grid = np.linspace(0, 1, 41)
W, R = np.meshgrid(grid, grid)                     # every (warmth, rain) on a 41 × 41 grid
OUT = relu(2.0 * W - 3.0 * R - 0.5)                 # the neuron, applied to all of them at once

fig = plt.figure(figsize=(7.8, 5.2))
ax = fig.add_subplot(projection="3d")
ax.set_title("every day, its output as height: the sunny day stands at 0.8, the rainy day at 0", fontsize=10)
ax.plot_surface(W, R, OUT, color=PAPER, edgecolor=INK, lw=0.35, shade=False, rstride=1, cstride=1)
ax.scatter([0.8], [0.1], [out_sunny], color=INK, s=60, depthshade=False)
ax.plot([0.8, 0.8], [0.6, 0.6], [0, 0.35], color=INK, lw=2)
ax.plot([0.8, 0.8], [0.1, 0.1], [out_sunny, 1.25], color=INK, lw=0.8)
ax.text(0.8, 0.1, 1.32, "sunny: 0.8", ha="center")
ax.set_xlabel("warmth"); ax.set_ylabel("rain"); ax.set_zlabel("output", labelpad=8)
for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
    pane.set_facecolor(PAPER); pane.set_edgecolor(INK2)
ax.set_zlim(0, 1.6); ax.set_box_aspect((1, 1, 0.75))
ax.view_init(elev=24, azim=-58)
plt.show()

print(f"warm, dry days fire hardest: at warmth 1, rain 0 the output is {OUT[0, -1]:.1f}")
check("the sunny day's column", float(OUT[4, 32]), 0.8, tol=1e-9)    # rain 0.1 is grid row 4, warmth 0.8 is column 32

## Plate XI · The neuron draws a line

Seen from above, one straight line splits all days in two: silent on one side, firing on the other, harder the further
you go. The line is where the total is exactly zero:

  2.0 × warmth − 3.0 × rain − 0.5 = 0

Below we hatch the side where the neuron fires and draw the line.

In [ ]:
def top_view(ax, w_warmth=2.0, w_rain=-3.0, bias=-0.5, label=True):
    """The neuron from above: the firing side hatched, the line where its total is 0."""
    g = np.linspace(0, 1, 201)
    W, R = np.meshgrid(g, g)
    T = w_warmth * W + w_rain * R + bias
    if T.max() > 0:
        ax.contourf(W, R, T, levels=[0, max(T.max(), 1e-6)], colors="none", hatches=["////"])
    ax.contour(W, R, T, levels=[0], colors=INK, linewidths=2.2)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    if label:
        ax.set_xlabel("warmth"); ax.set_ylabel("rain")


fig, ax = plt.subplots(figsize=(4.2, 4.2))
top_view(ax)
ax.text(0.68, 0.12, "fires", ha="center"); ax.text(0.3, 0.7, "silent", ha="center")
ax.plot([0.8], [0.1], "o", color=INK); ax.plot([0.8], [0.6], "o", mfc=PAPER, mec=INK)
plt.show()

# where the line crosses the edges of the square: solve the total = 0 for one input, with the other fixed
check("on dry days (rain 0), firing starts at warmth", 0.5 / 2.0, 0.25)
check("on the warmest days (warmth 1), firing stops at rain", (2.0 - 0.5) / 3.0, 0.5)

## Plates XII–XIII · Weights turn the line; the bias slides it

Make rain count for more, from −0.75 down to −3.0, and the line swings until only fairly dry days get through. Change the
bias instead and the line slides without turning.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, w in zip(axes, (-0.75, -1.5, -2.25, -3.0)):
    top_view(ax, w_rain=w, label=False)
    ax.set_title(minus(f"weight on rain {w:+.2f}"))
axes[0].set_ylabel("rain")
for ax in axes: ax.set_xlabel("warmth")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (title, w2, b) in zip(axes, (("our neuron", -3.0, -0.5), ("rain counts only × −1", -1.0, -0.5),
                                     ("bias −1.2: harder to convince", -3.0, -1.2))):
    top_view(ax, w_rain=w2, bias=b, label=False)
    ax.set_title(title, fontsize=10)
plt.show()

# a weight turns the line (it now meets the top edge), the bias slides it (it meets the bottom edge further right)
check("rain × −1: on the wettest days (rain 1), firing starts at warmth", (1.0 + 0.5) / 2.0, 0.75)
check("bias −1.2: on dry days, firing starts at warmth", 1.2 / 2.0, 0.6)

## Plate XIV · A diary of days

Suppose you kept a diary of 42 walks: a solid dot for a walk you enjoyed, a hollow dot for one you did not. (It is made up.)
The lecture drew these exact dots with Python's random number generator, seed 11. Python's generator gives the same
numbers on every computer, so we can make them again here, not download them.

The diary keeps 18 enjoyed days and 24 others, and leaves out days too close to the neuron's line to call.

In [ ]:
import random

def make_diary(rule, seed, good, bad):
    """Draw random days until we have `good` enjoyed and `bad` not; rule(w, r) says which (None: too close to call)."""
    rng = random.Random(seed)
    days, need = [], {True: good, False: bad}
    while need[True] or need[False]:
        w, r = rng.uniform(0.03, 0.97), rng.uniform(0.03, 0.97)
        label = rule(w, r)
        if label is None or not need[label]:
            continue
        need[label] -= 1
        days.append((w, r, label))
    return days

def walk_rule(w, r):
    t, _ = neuron(w, r)
    return None if abs(t) < 0.18 else t > 0

diary = make_diary(walk_rule, seed=11, good=18, bad=24)
check("days in the diary", len(diary), 42)
check("enjoyed days", sum(d[2] for d in diary), 18)
check("the first day, as the lecture drew it", (round(diary[0][0], 4), round(diary[0][1], 4), diary[0][2]), (0.4552, 0.5562, False))

In [ ]:
def draw_diary(ax, days):
    for w, r, enjoyed in days:
        ax.plot(w, r, "o", ms=7, mfc=INK if enjoyed else PAPER, mec=INK, mew=1.2)

fig, ax = plt.subplots(figsize=(4.4, 4.4))
top_view(ax)
draw_diary(ax, diary)
plt.show()

fires = [neuron(w, r)[1] > 0 for w, r, _ in diary]
right = sum(f == enjoyed for f, (_, _, enjoyed) in zip(fires, diary))
check("days where 'fires' matches 'enjoyed'", right, 42)

Every enjoyed day lies on the firing side, and every other day on the silent side. Our neuron *works* on this diary.

### Your turn 2 · Break it

Add one made-up day to the diary that the neuron gets wrong: an enjoyed day on the silent side, or the other way round.

*Hint:* a hollow (not enjoyed) day anywhere in the hatched region will do.

In [ ]:
my_day = None   # TODO: (warmth, rain, enjoyed) for a day the neuron gets wrong, e.g. (0.9, 0.05, False)

## Plate XV · Reading the neuron

We can read this neuron completely:

- **+2.0 on warmth**: it likes warm days;
- **−3.0 on rain**: it dislikes rain, more than it likes warmth;
- **−0.5 bias**: it needs some convincing.

Nothing is hidden. That is interpretability at its simplest, and exactly what gets hard with billions of neurons.

## Plate XVIII · Check yourself

The lecture asked three questions. Here they are as exercises.

### Your turn 3 · Two more days

What does the neuron output on a day with warmth 0.5 and rain 0? And on a chilly, dry day, warmth 0.2 and rain 0?

*Hint:* call `neuron(warmth, rain)`; the second number it returns is the output.

In [ ]:
out_mild = None    # TODO: output for warmth 0.5, rain 0
out_chilly = None  # TODO: output for warmth 0.2, rain 0

### Your turn 4 · Stop caring about rain

Which one number would you change so the neuron stops caring about rain? Change it, and draw the neuron from above with
`top_view`. Where does its line sit now?

*Hint:* `top_view(ax, w_rain=...)`.

In [ ]:
# TODO: draw the neuron that ignores rain

## Plate XIX · The limit of one

Now suppose you only enjoy walks when it is **neither too cold nor too hot** (and not too wet). The enjoyed days sit in a
band. This diary was drawn with seed 5: 16 enjoyed days, 26 others.

Can any one straight line split it? We try every direction (every whole degree) and every position (steps of 0.01), as
the lecture's figure does, and keep the line that gets the most days right.

In [ ]:
def band_rule(w, r):
    inside = 0.3 < w < 0.68 and r < 0.55
    edge = min(abs(w - 0.3), abs(w - 0.68)) < 0.05 or abs(r - 0.55) < 0.05
    return None if edge else inside

band = make_diary(band_rule, seed=5, good=16, bad=26)
check("enjoyed days in the band diary", sum(d[2] for d in band), 16)
check("its first day, as the lecture drew it", (round(band[0][0], 4), round(band[0][1], 4), band[0][2]), (0.6155, 0.7273, False))

X = np.array([(w, r) for w, r, _ in band])
Y = np.array([e for *_, e in band])
offsets = np.round(np.arange(-1.5, 1.5 + 1e-9, 0.01), 2)
best_right, best_line = -1, None
for deg in range(360):
    a = np.deg2rad(deg)
    w1, w2 = 2 * np.cos(a), 2 * np.sin(a)                # the line's direction
    fires = (X @ [w1, w2])[None, :] - 2 * offsets[:, None] > 0      # every position at once
    right = (fires == Y[None, :]).sum(axis=1)
    i = int(right.argmax())
    if right[i] > best_right:
        best_right, best_line = int(right[i]), (w1, w2, -2 * offsets[i])

print(f"the best single line gets {best_right} of 42 right, {42 - best_right} wrong")
check("the best any straight line can do", best_right, 35)

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4.4))
top_view(ax, *best_line)
ax.add_patch(plt.Rectangle((0.3, 0), 0.38, 0.55, fill=False, ls="--", ec=INK2, lw=1))
draw_diary(ax, band)
wrong = [(w, r) for (w, r, e) in band if (best_line[0] * w + best_line[1] * r + best_line[2] > 0) != e]
for w, r in wrong:
    ax.plot(w, r, "o", ms=15, mfc="none", mec=INK, mew=1, ls="none")
ax.set_title("the best line, its 7 wrong days ringed", fontsize=10)
plt.show()

One neuron draws one straight line, and some patterns need more than one. That is where Lecture 3 picks up. First,
Lecture 2 asks how a neuron finds its weights on its own.

### Your turn 5 · Try to beat the best line

Pick your own weights and bias and count how many band days your neuron gets right. Can you beat 35?

*Hint:* a day is "right" when (total > 0) matches enjoyed. You can't beat 35: the search above tried every line.

In [ ]:
my_weights = None   # TODO: (w_warmth, w_rain, bias), then count the band days your line gets right

## What we built

- A neuron weighs its inputs, adds a bias, and passes the total through a gate.
- Over all inputs it draws one straight line: silent on one side, firing on the other.
- Weights turn the line; the bias slides it.
- One neuron can be read completely, and one neuron is not enough for a band.

**Next · Lecture 2, Learning:** we set these weights by hand. How does a network find them on its own?

---

## Solutions

**Your turn 1.** 0.8 × 2.0 − 3.0 × rain − 0.5 = 0 gives rain = 1.1 ÷ 3.

In [ ]:
rain_edge = (0.8 * 2.0 - 0.5) / 3.0
print(f"the neuron fires again once rain drops below {rain_edge:.4f}")
check("just below the edge it fires", neuron(0.8, rain_edge - 0.01)[1] > 0, True)
check("just above the edge it is silent", neuron(0.8, rain_edge + 0.01)[1], 0.0)

**Your turn 2.** A hollow day deep in the firing side.

In [ ]:
my_day = (0.9, 0.05, False)
total, out = neuron(my_day[0], my_day[1])
print(f"total {total:+.2f}: the neuron fires, but this day was not enjoyed")
check("the neuron gets my day wrong", (out > 0) != my_day[2], True)

**Your turn 3.** The lecture's answers: 0.5 × 2.0 − 0.5 = 0.5, and 0.2 × 2.0 − 0.5 = −0.1, so 0.

In [ ]:
out_mild = neuron(0.5, 0.0)[1]
out_chilly = neuron(0.2, 0.0)[1]
check("warmth 0.5, rain 0", out_mild, 0.5)
check("warmth 0.2, rain 0", out_chilly, 0.0)

**Your turn 4.** Set the weight on rain to 0. The line stands upright at warmth 0.25: only warmth matters now.

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 3.6))
top_view(ax, w_rain=0.0)
ax.set_title("weight on rain 0", fontsize=10)
plt.show()
check("the line, at any rain, sits at warmth", 0.5 / 2.0, 0.25)

**Your turn 5.** For example a steep line between cool and warm days. Nothing beats 35.

In [ ]:
my_weights = (-2.0, -2.0, 2.0)
w1, w2, b = my_weights
mine = sum(((w1 * w + w2 * r + b) > 0) == e for w, r, e in band)
print(f"my line gets {mine} of 42 right")
check("no line beats the best", mine <= best_right, True)